# Colibrimo: room video to 3D scene (Google Colab)

Runs the Colibrimo pipeline (`cpipe`) on a free Colab GPU:
video, then frames, then camera poses (COLMAP), then Gaussian Splatting training (gsplat), then `scene.ply`.

**Before you start**
1. *Runtime > Change runtime type >* **T4 GPU**.
2. Put your video in Google Drive, e.g. `MyDrive/colibrimo/videos/room1.mp4`.
3. Edit the settings cell below, then *Runtime > Run all*.

**Privacy:** room videos are personal data. Only use your own test rooms here until Colibrimo
approves storing client videos on Google Drive.

**Time:** the first session compiles two GPU libraries (about 25 min) and caches them on your
Drive; later sessions install in about 2 min. Training takes roughly 10 to 60 min depending on
the preset. Results are copied to `MyDrive/colibrimo/runs/<RUN_NAME>/`.

Full documentation: `docs/pipeline.md` in the repository.

In [ ]:
# ---- Settings: edit these -------------------------------------------------
VIDEO = "/content/drive/MyDrive/colibrimo/videos/room1.mp4"  # your video on Drive
RUN_NAME = "room1"     # results: MyDrive/colibrimo/runs/<RUN_NAME>/
PRESET = "fast"        # fast | balanced | quality
EXTRA_ARGS = ""        # e.g. "--steps 2000", "--force", "--from-step train"
PIPELINE_REF = "main"  # branch or tag of the Colibrimo repository to install

## 1. Check the GPU and connect Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive

drive.mount("/content/drive")

## 2. Install

- COLMAP and FFmpeg from Ubuntu packages.
- gsplat and fused-ssim are compiled once for this Colab setup and cached on your Drive
  (`MyDrive/colibrimo/wheels/`). When Colab upgrades Python or PyTorch, they are compiled again.
- gsplat's training script (`examples/simple_trainer.py`, gsplat v1.5.3) and its requirements.
- The Colibrimo pipeline itself, from GitHub.

In [ ]:
import glob
import os
import subprocess
import sys

import torch

GSPLAT_VERSION = "1.5.3"
# fused-ssim commit pinned by gsplat v1.5.3 (examples/requirements.txt)
FUSED_SSIM = "git+https://github.com/rahul-goel/fused-ssim@328dc9836f513d00c4b5bc38fe30478b4435cbb5"
REPO = "https://github.com/ALpatson/colibromo_yes_project_4006.git"


def sh(cmd):
    """Run a shell command, show its output here, stop the notebook if it fails."""
    print("$", cmd)
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"Command failed (exit code {proc.returncode}): {cmd}")


# 1. System tools
sh("apt-get -qq update && apt-get -qq install -y colmap ffmpeg > /dev/null")

# 2. GPU libraries: compile once, cache on Drive
arch = "".join(map(str, torch.cuda.get_device_capability()))
tag = f"py{sys.version_info.major}{sys.version_info.minor}-torch{torch.__version__.replace('+', '-')}-sm{arch}"
cache = f"/content/drive/MyDrive/colibrimo/wheels/{tag}"
os.makedirs(cache, exist_ok=True)
os.environ["MAX_JOBS"] = "2"  # more parallel jobs run out of RAM on Colab
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{arch[0]}.{arch[1:]}"  # only this GPU: faster build


def install_cached(name, source):
    if not glob.glob(f"{cache}/{name}*.whl"):
        print(f"Compiling {name} (first time for this Colab setup, can take 20+ min)...")
        sh(f'pip wheel -q --no-build-isolation --no-deps -w "{cache}" "{source}"')
    sh(f'pip install -q "{glob.glob(f"{cache}/{name}*.whl")[0]}"')


sh("pip install -q ninja")
install_cached("gsplat", f"git+https://github.com/nerfstudio-project/gsplat.git@v{GSPLAT_VERSION}")
install_cached("fused_ssim", FUSED_SSIM)

# 3. gsplat's training script + its requirements (numpy pin and the GPU
#    packages above are left out: Colab's numpy 2 works, see docs/decisions.md)
if not os.path.isdir("/content/gsplat"):
    sh(f"git clone -q --depth 1 --branch v{GSPLAT_VERSION} https://github.com/nerfstudio-project/gsplat.git /content/gsplat")
sh("grep -vE 'numpy|fused-ssim|fused-bilagrid|^#|^\\s*$' /content/gsplat/examples/requirements.txt > /content/gsplat_requirements.txt")
sh("pip install -q -r /content/gsplat_requirements.txt")
os.environ["GSPLAT_EXAMPLES_DIR"] = "/content/gsplat/examples"

# 4. The Colibrimo pipeline
sh(f'pip install -q --force-reinstall --no-deps "colibrimo-pipeline @ git+{REPO}@{PIPELINE_REF}#subdirectory=pipeline"')
sh("cpipe --version")

## 3. Run the pipeline

The run happens on Colab's local disk (fast) and is then copied to Drive.
If `MyDrive/colibrimo/runs/<RUN_NAME>/` already exists (for example camera poses computed on a
laptop with `--until poses`), it is copied in first and completed steps are skipped.

In [ ]:
local_run = f"/content/runs/{RUN_NAME}"
drive_run = f"/content/drive/MyDrive/colibrimo/runs/{RUN_NAME}"
os.makedirs(local_run, exist_ok=True)
if os.path.isdir(drive_run):
    sh(f'rsync -a "{drive_run}/" "{local_run}/"')

# Stream cpipe's messages into this cell (Colab doesn't show a plain
# subprocess's output).
proc = subprocess.Popen(
    f'cpipe run --video "{VIDEO}" --output "{local_run}" --preset {PRESET} {EXTRA_ARGS}',
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in proc.stdout:
    print(line, end="")
result = proc.wait()

# Save to Drive even if the run failed (logs help debugging).
os.makedirs(drive_run, exist_ok=True)
sh(f'rsync -a --exclude frames_candidates "{local_run}/" "{drive_run}/"')
print("Saved to", drive_run)
if result != 0:
    print(f"\nThe run FAILED. Read the message above, or the logs in {drive_run}/logs/")

## 4. Results

In [ ]:
import json

manifest = json.load(open(f"{local_run}/manifest.json"))
print("Status:", manifest["status"])
for name, step in manifest["steps"].items():
    print(f"  {name:15s} {step['status']:10s} {step.get('duration_s', 0):8.0f} s")
if "scene_ply" in manifest.get("outputs", {}):
    size_mb = os.path.getsize(f"{local_run}/scene.ply") / 1e6
    print(f"\nscene.ply ({size_mb:.0f} MB) is in {drive_run}/")

**Look at the result:** download `scene.ply` from Drive and drop it on
[SuperSplat](https://superspl.at/editor) (runs in your browser; the file is not uploaded).
Our own viewer comes in Step 4.